# 04 - Evaluation: Quality scoring and grading

This notebook exercises the rule-based grading layer defined in
`task2_3_4_cv_quality/src/grading.py`.

## Heuristic overview

The grading layer runs **after** the classifier. It consumes a clean,
post-processed prediction dictionary of the form:

```python
{"predicted_class": "freshapples", "confidence": 0.91}
```

and returns three proxy quality percentages on a 0-100 scale plus a
letter grade and a recommendation. The heuristic is:

- **`fresh*`** classes
  - `color_score` and `ripeness_score` track confidence directly (`conf * 100`).
  - `size_score` is moderately high and less sensitive (`70 + conf * 30`).
- **`rotten*`** classes
  - higher confidence means the model is more sure the item is rotten, so
    quality scores **decrease** as confidence rises.
  - `color_score` and `ripeness_score` are harshly reduced; `size_score`
    is reduced more gently.

Grade = `A` if `min(scores) >= 80`, `B` if `>= 60`, otherwise `C`. Taking
the minimum means a single weak dimension pulls the grade down.

> **Note.** These are *proxy* quality scores derived from the predicted
> class and classifier confidence. They are **not** direct sensor
> measurements of real colour, size or ripeness.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "config.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from task2_3_4_cv_quality.src.grading import (
    assign_grade,
    compute_quality_scores,
    get_recommendation,
    grade_produce,
)

## Dummy examples

Four hand-picked cases cover both freshness states at high and medium
confidence levels.

In [3]:
examples = [
    {"predicted_class": "Apple__Healthy",   "confidence": 0.98},
    {"predicted_class": "Banana__Rotten",   "confidence": 0.92},
    {"predicted_class": "Tomato__Rotten",   "confidence": 0.55},
]

results = [grade_produce(ex) for ex in examples]
for r in results:
    print(r)

{'predicted_class': 'Apple__Healthy', 'produce_type': 'Apple', 'state': 'Healthy', 'confidence': 0.98, 'color_score': 98, 'size_score': 99, 'ripeness_score': 98, 'grade': 'A', 'recommendation': 'Premium quality - sell at full price in premium display'}
{'predicted_class': 'Banana__Rotten', 'produce_type': 'Banana', 'state': 'Rotten', 'confidence': 0.92, 'color_score': 3, 'size_score': 52, 'ripeness_score': 3, 'grade': 'C', 'recommendation': 'Low quality - heavy discount or remove from sale'}
{'predicted_class': 'Tomato__Rotten', 'produce_type': 'Tomato', 'state': 'Rotten', 'confidence': 0.55, 'color_score': 18, 'size_score': 59, 'ripeness_score': 16, 'grade': 'C', 'recommendation': 'Low quality - heavy discount or remove from sale'}


## Tabular view

Same results rendered as a compact table for easy comparison.

In [4]:
header = ["predicted_class", "confidence", "color", "size", "ripeness", "grade", "recommendation"]
widths = [18, 10, 5, 5, 8, 5, 60]

def _fmt_row(values):
    return "  ".join(f"{str(v):<{w}}" for v, w in zip(values, widths))

print(_fmt_row(header))
print("-" * (sum(widths) + 2 * (len(widths) - 1)))
for r in results:
    print(_fmt_row([
        r["predicted_class"],
        f"{r['confidence']:.2f}",
        r["color_score"],
        r["size_score"],
        r["ripeness_score"],
        r["grade"],
        r["recommendation"],
    ]))

predicted_class     confidence  color  size   ripeness  grade  recommendation                                              
---------------------------------------------------------------------------------------------------------------------------
Apple__Healthy      0.98        98     99     98        A      Premium quality - sell at full price in premium display     
Banana__Rotten      0.92        3      52     3         C      Low quality - heavy discount or remove from sale            
Tomato__Rotten      0.55        18     59     16        C      Low quality - heavy discount or remove from sale            


## Helper functions in isolation

The intermediate functions can also be called directly.

In [6]:
sample = {"predicted_class": "Orange__Healthy", "confidence": 0.84}
scores = compute_quality_scores(sample)
print("scores        :", scores)
grade = assign_grade(scores)
print("grade         :", grade)
print("recommendation:", get_recommendation(grade))

scores        : {'predicted_class': 'Orange__Healthy', 'produce_type': 'Orange', 'state': 'Healthy', 'confidence': 0.84, 'color_score': 84, 'size_score': 95, 'ripeness_score': 84}
grade         : A
recommendation: Premium quality - sell at full price in premium display


## Input validation

Malformed inputs raise clear exceptions so integration bugs surface
early.

In [8]:
bad_inputs = [
    {"predicted_class": "Apple__Healthy"},
    
    {"predicted_class": "Apple__Healthy", "confidence": 1.7},
    

    {"predicted_class": "", "confidence": 0.5},
    {"predicted_class": "mystery_item", "confidence": 0.9},
    
    {"predicted_class": "freshapples", "confidence": 0.8},
    
    {"predicted_class": "Apple__Moldy", "confidence": 0.7},
    
    "not a dict",
]

print("Testing error handling:\n")
for i, bad in enumerate(bad_inputs, 1):
    try:
        grade_produce(bad)
        print(f"Test {i}: ❌ Should have raised an error!")
    except Exception as exc:
        print(f"Test {i}: {type(exc).__name__:<15} → {exc}")

Testing error handling:

Test 1: KeyError        → "model_output missing 'confidence'"
Test 2: ValueError      → Confidence must be in [0, 1], got 1.7
Test 3: ValueError      → Invalid class name format: ''. Expected format: 'ProduceType__State' (e.g., 'Apple__Healthy')
Test 4: ValueError      → Invalid class name format: 'mystery_item'. Expected format: 'ProduceType__State' (e.g., 'Apple__Healthy')
Test 5: ValueError      → Invalid class name format: 'freshapples'. Expected format: 'ProduceType__State' (e.g., 'Apple__Healthy')
Test 6: ValueError      → Invalid state: 'Moldy'. Expected 'Healthy' or 'Rotten'.
Test 7: KeyError        → "model_output missing 'predicted_class'"


## Summary

- Scores are derived **only** from the predicted class + confidence; no
  image-level measurements are made here.
- Grade A/B/C comes from the minimum of the three scores, so a single
  weak dimension can downgrade an otherwise strong item.
- Recommendations are keyed off the grade and can be overridden by
  editing `RECOMMENDATIONS` in `grading.py`.